# Module 06 — Java GC & Gluten

Part A: JVM tuning. Part B: optional Gluten native acceleration.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import get_spark, ui_links
from shared import config
from shared.utils import timed_action

spark = get_spark('module-06-java-gc')
ui_links()

## Part A — JVM / GC Tuning

Driver and executor JVMs are separate processes. GC pauses show up in Spark UI **Executors** tab (GC Time column).

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())

# Memory-heavy job: wide aggregations trigger object allocation
heavy = trips.groupBy('PULocationID', 'DOLocationID', 'VendorID').count()
_, t = timed_action(heavy)
print(f'Aggregation completed in {t:.2f}s')
print('Check Executors tab → GC Time column at http://localhost:4040')

### Current JVM Settings

Inspect executor memory and GC config:

In [ ]:
for item in ['spark.driver.memory', 'spark.executor.memory', 'spark.memory.fraction']:
    print(f'{item}: {spark.conf.get(item, "default")}')

### GC Strategies (Java 17+)

| GC | Flag | Best for |
|----|------|----------|
| G1 (default) | `-XX:+UseG1GC` | General purpose |
| ZGC | `-XX:+UseZGC` | Low-latency, large heaps |

To experiment, set in `docker/spark-defaults.conf`:
```
spark.executor.extraJavaOptions  -XX:+UseG1GC -XX:G1HeapRegionSize=16m
spark.driver.extraJavaOptions    -XX:+UseG1GC
```
Then restart the cluster.

### Key Memory Configs

- `spark.memory.fraction` (default 0.6) — fraction of heap for Spark vs user memory
- `spark.memory.storageFraction` — cache vs execution memory split
- OOM symptoms: tasks fail with `OutOfMemoryError`, spill to disk increases

## Part B — Gluten (Optional)

Gluten offloads Spark operators to native engines (Velox). Requires separate setup.

> **Note:** Gluten 1.7.0 supports Spark 4.1 but excludes Iceberg in preview builds. The Gluten demo uses plain Parquet.

In [ ]:
import os
GLUTEN_ENABLED = os.environ.get('SPARK_GLUTEN_ENABLED', 'false').lower() == 'true'
print(f'Gluten enabled: {GLUTEN_ENABLED}')

if not GLUTEN_ENABLED:
    print('Start with: make up-gluten (after installing Gluten binaries — see README)')
else:
    bench = spark.read.parquet(*config.yellow_taxi_paths())
    result = bench.groupBy('PULocationID').agg({'total_amount': 'sum', 'trip_distance': 'avg'})
    result.explain('formatted')
    _, t = timed_action(result)
    print(f'Gluten benchmark: {t:.2f}s')

## Key Takeaways

1. Monitor GC Time in Spark UI before tuning JVM flags.
2. Increase executor memory or partitions before reaching for exotic GC settings.
3. Gluten can accelerate CPU-bound SQL but requires compatible native libraries and excludes Iceberg today.